# Direct Preference Optimization (DPO) with TRL & Qwen2.5-1.5B-Instruct

This notebook implements a clean, production-grade Direct Preference Optimization (DPO) pipeline using Hugging Face's **TRL (`DPOTrainer`)** and **PEFT (LoRA)**.

### Pipeline Overview:
1. **Install & Versions**: Verify compatible `transformers`, `trl`, and `peft` versions.
2. **Model & Tokenizer**: Load `Qwen/Qwen2.5-1.5B-Instruct`.
3. **Dataset**: Load `HuggingFaceH4/ultrafeedback_binarized` (conversational preference format).
4. **LoRA & DPOConfig**: Configure target projection modules and memory-efficient training arguments.
5. **DPOTrainer**: Leverage native TRL reference-model handling (`ref_model=None`) and chat templating.
6. **Training**: Train policy adapter using DPO objective.
7. **Saving & Verification**: Save LoRA adapter and verify artifacts.
8. **TRL Metrics**: Inspect evaluation metrics (`eval_loss`, reward margins, accuracies).
9. **Base vs DPO Evaluation**: Compare preference accuracy over 1000+ test samples using separate model instances.

In [ ]:
!pip install -q -U transformers datasets accelerate peft trl "torchao>=0.16.0"

In [ ]:
import os
import gc
import torch
import torch.nn.functional as F
import transformers
import trl
import peft
import datasets

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from trl import DPOTrainer, DPOConfig

# Version handling
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Initial VRAM: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")

## 1. Load Tokenizer & Model Configuration

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print(f"Loaded tokenizer for {MODEL_NAME}")
print("Chat template available:", tokenizer.chat_template is not None)

## 2. Load Conversational Preference Dataset

The `HuggingFaceH4/ultrafeedback_binarized` dataset already contains conversational `chosen` and `rejected` message pairs. `DPOTrainer` processes these natively through the tokenizer's chat template.

In [ ]:
dataset = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="train_prefs"
)
eval_dataset = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="test_prefs"
)

print(f"Train examples: {len(dataset)}")
print(f"Eval examples:  {len(eval_dataset)}")

# Inspect first sample
sample = dataset[0]
print("\nFeatures:", list(sample.keys()))
print("Prompt:", sample["prompt"][:120], "...")
print("Chosen turns:", len(sample["chosen"]))
print("Rejected turns:", len(sample["rejected"]))

## 3. LoRA Configuration & DPOConfig

- We set up PEFT `LoraConfig` targeting attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`).
- We pass `peft_config` directly to `DPOTrainer` without manual `get_peft_model()` wrapping.
- Memory optimizations: `fp16=True`, `gradient_checkpointing=True`, batch size 1 with 8 gradient accumulation steps (target: Tesla T4 / ~16 GB VRAM).

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

# Adapt eval_strategy vs evaluation_strategy based on transformers version
eval_strategy_key = "eval_strategy" if hasattr(transformers.TrainingArguments, "eval_strategy") else "evaluation_strategy"

dpo_config_kwargs = {
    "output_dir": "./qwen15b_dpo",
    "beta": 0.1,
    "learning_rate": 1e-5,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "logging_steps": 10,
    eval_strategy_key: "steps",
    "eval_steps": 100,
    "save_strategy": "steps",
    "save_steps": 100,
    "save_total_limit": 2,
    "warmup_ratio": 0.05,
    "fp16": (device == "cuda"),
    "gradient_checkpointing": True,
    "max_length": 512,
    "max_prompt_length": 256,
    "report_to": "none",
}

training_args = DPOConfig(**dpo_config_kwargs)
print("DPOConfig configured successfully.")

## 4. Initialize DPOTrainer & Pre-Training Diagnostics

We pass `ref_model=None`. When `peft_config` is provided, TRL automatically disables LoRA adapter adapters during the reference forward pass, avoiding the need to load an extra 1.5B reference model.

In [ ]:
# Instantiate DPOTrainer with compatibility for processing_class / tokenizer
trainer_kwargs = {
    "model": MODEL_NAME,
    "ref_model": None,
    "args": training_args,
    "train_dataset": dataset,
    "eval_dataset": eval_dataset,
    "peft_config": peft_config,
}

try:
    trainer = DPOTrainer(
        **trainer_kwargs,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = DPOTrainer(
        **trainer_kwargs,
        tokenizer=tokenizer,
    )

# Pre-training diagnostics
trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in trainer.model.parameters())
trainable_pct = 100 * trainable_params / total_params

print("=" * 60)
print("PRE-TRAINING DIAGNOSTICS")
print("=" * 60)
print(f"Model:                {MODEL_NAME}")
print(f"Train examples:       {len(dataset)}")
print(f"Eval examples:        {len(eval_dataset)}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable %:          {trainable_pct:.4f}%")
print("=" * 60)

## 5. Execute DPO Training

In [ ]:
train_result = trainer.train()
print("\nTraining complete!")
print(f"Final Training Loss: {train_result.training_loss:.4f}")

## 6. Save Model & Verify LoRA Adapter Files

In [ ]:
OUTPUT_DIR = "./qwen15b_dpo"
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved trained model and tokenizer to '{OUTPUT_DIR}'")

# Verify saved LoRA adapter files
saved_files = os.listdir(OUTPUT_DIR)
print("\nVerifying saved output files:")
for f in sorted(saved_files):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / (1024 * 1024)
    print(f" - {f:30s} ({size_mb:.2f} MB)")

has_adapter = any("adapter" in f for f in saved_files)
if has_adapter:
    print("\n[OK] LoRA adapter files successfully verified in output directory.")
else:
    print("\n[WARNING] Adapter files not found directly; check saved directory structure.")

## 7. TRL Post-Training Evaluation Metrics

Extract evaluation loss, chosen rewards, rejected rewards, accuracy, and reward margins directly from `trainer.evaluate()`.

In [ ]:
eval_metrics = trainer.evaluate()

print("=" * 60)
print("TRL DPO EVALUATION METRICS")
print("=" * 60)
for k, v in eval_metrics.items():
    if isinstance(v, float):
        print(f"{k:30s}: {v:.6f}")
    else:
        print(f"{k:30s}: {v}")
print("=" * 60)

## 8. Base Model vs DPO Model Preference Accuracy (1000+ Examples)

We load a fresh base model separately, and load the saved LoRA adapter onto a separate base model instance to ensure isolation. Both models are evaluated under the exact same scoring convention on $\ge 1000$ preference pairs.

In [ ]:
# Free trainer resources from GPU memory
del trainer
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# 1. Pure Base Model
print("Loading fresh Base Model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
if device != "cuda":
    base_model = base_model.to(device)
base_model.eval()

# 2. DPO Model (Separate Base Model + LoRA Adapter)
print("Loading DPO Model (Base + LoRA Adapter)...")
dpo_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
if device != "cuda":
    dpo_base = dpo_base.to(device)

dpo_model = PeftModel.from_pretrained(
    dpo_base,
    OUTPUT_DIR,
)
dpo_model.eval()

print("Models loaded independently for evaluation.")

In [ ]:
def get_response_logprob(model, tokenizer, conversation, max_length=512):
    """
    Computes summed log-probability of the assistant response tokens given the conversation.
    """
    full_text = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=False
    )
    
    prompt_messages = conversation[:-1]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    full_tokens = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(model.device)
    
    prompt_tokens = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(model.device)
    
    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]
    
    prompt_len = prompt_tokens["input_ids"].shape[1]
    seq_len = input_ids.shape[1]
    
    if prompt_len >= seq_len:
        return torch.tensor(0.0, device=model.device)
        
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :].contiguous()
    labels = input_ids[:, 1:].contiguous()
    
    token_log_probs = -F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        labels.view(-1),
        reduction="none"
    ).view_as(labels)
    
    response_mask = torch.zeros_like(token_log_probs, dtype=torch.bool)
    response_mask[:, (prompt_len - 1):] = True
    response_mask &= attention_mask[:, 1:].bool()
    
    return (token_log_probs * response_mask).sum(dim=1).squeeze()


def evaluate_preference_accuracy(model, tokenizer, eval_data, num_examples=1000, max_length=512):
    """
    Calculates preference accuracy: proportion of samples where log P(chosen) > log P(rejected).
    """
    model.eval()
    correct = 0
    total = min(num_examples, len(eval_data))
    eval_subset = eval_data.select(range(total))
    
    with torch.inference_mode():
        for i, sample in enumerate(eval_subset):
            chosen_lp = get_response_logprob(model, tokenizer, sample["chosen"], max_length=max_length)
            rejected_lp = get_response_logprob(model, tokenizer, sample["rejected"], max_length=max_length)
            
            if chosen_lp.item() > rejected_lp.item():
                correct += 1
                
            if (i + 1) % 250 == 0 or (i + 1) == total:
                print(f"Evaluated {i + 1:4d}/{total} samples | Current Accuracy: {correct / (i + 1):.2%}")
                
    return correct / total

In [ ]:
NUM_EVAL_SAMPLES = 100

print(f"--- Evaluating Pure Base Model ({NUM_EVAL_SAMPLES} samples) ---")
base_accuracy = evaluate_preference_accuracy(
    base_model,
    tokenizer,
    eval_dataset,
    num_examples=NUM_EVAL_SAMPLES,
    max_length=512
)

print(f"\n--- Evaluating DPO Policy Model ({NUM_EVAL_SAMPLES} samples) ---")
dpo_accuracy = evaluate_preference_accuracy(
    dpo_model,
    tokenizer,
    eval_dataset,
    num_examples=NUM_EVAL_SAMPLES,
    max_length=512
)

improvement = dpo_accuracy - base_accuracy

print("\n" + "=" * 60)
print("FINAL PREFERENCE ACCURACY COMPARISON")
print("=" * 60)
print(f"Evaluation pairs: {NUM_EVAL_SAMPLES}")
print(f"Base accuracy:    {base_accuracy:.2%}")
print(f"DPO accuracy:     {dpo_accuracy:.2%}")
print(f"Improvement:      {improvement:+.2%}")
print("=" * 60)